In [1]:
import cv2
import mediapipe as mp
import numpy as np

# --- 1. Dynamic Time Warping (DTW) for Sequence Alignment ---
# The KSI formula requires comparing frame N of the user to the "equivalent" frame N of the expert.
# DTW finds this optimal alignment between two sequences of different lengths.

def dynamic_time_warping(seq1, seq2):
    """
    Computes the optimal alignment and distance between two sequences using DTW.
    Args:
        seq1 (np.array): First sequence of shape (n_frames, n_dims).
        seq2 (np.array): Second sequence of shape (m_frames, n_dims).
    Returns:
        (np.array, np.array): Aligned sequences seq1_aligned, seq2_aligned.
    """
    n, m = len(seq1), len(seq2)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = np.linalg.norm(seq1[i - 1] - seq2[j - 1]) # Euclidean distance
            last_min = min(dtw_matrix[i-1, j], dtw_matrix[i, j-1], dtw_matrix[i-1, j-1])
            dtw_matrix[i, j] = cost + last_min

    # Traceback to find the optimal path
    path = []
    i, j = n, m
    while i > 0 and j > 0:
        path.append((i-1, j-1))
        i, j = min((i-1, j), (i, j-1), (i-1, j-1), key=lambda x: dtw_matrix[x[0], x[1]])

    path.reverse()
    
    # Create aligned sequences
    seq1_aligned = np.array([seq1[i] for i, j in path])
    seq2_aligned = np.array([seq2[j] for i, j in path])

    return seq1_aligned, seq2_aligned



2025-09-24 23:18:08.110022: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-24 23:18:08.245223: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758736088.292884  178036 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758736088.306474  178036 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758736088.413174  178036 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
# --- 2. Kinetic Similarity Index (KSI) Implementation ---

def calculate_ksi(expert_seq, user_seq, weights={'pose': 0.4, 'velocity': 0.4, 'acceleration': 0.2}, alpha=0.1, beta=0.1):
    """
    Calculates the Kinetic Similarity Index (KSI) between two motion sequences.
    """
    # Step 1: Align sequences using DTW
    expert_aligned, user_aligned = dynamic_time_warping(expert_seq, user_seq)
    
    # Helper function for cosine similarity
    def cosine_similarity(v1, v2):
        # Add a small epsilon to avoid division by zero if a vector is all zeros
        epsilon = 1e-8
        return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + epsilon)

    # --- Component 2a: Pose Similarity (S_pose) ---
    # Using vectors for upper arm (shoulder to elbow) and forearm (elbow to wrist)
    # Expert vectors
    v_upper_arm_exp = expert_aligned[:, 1] - expert_aligned[:, 0]  # Elbow - Shoulder
    v_forearm_exp = expert_aligned[:, 2] - expert_aligned[:, 1]    # Wrist - Elbow
    # User vectors
    v_upper_arm_user = user_aligned[:, 1] - user_aligned[:, 0]
    v_forearm_user = user_aligned[:, 2] - user_aligned[:, 1]
    
    pose_sims = []
    for i in range(len(v_upper_arm_exp)):
        upper_arm_sim = cosine_similarity(v_upper_arm_exp[i], v_upper_arm_user[i])
        forearm_sim = cosine_similarity(v_forearm_exp[i], v_forearm_user[i])
        # Average the similarity of the two arm segments for this frame
        pose_sims.append((upper_arm_sim + forearm_sim) / 2)
    s_pose = np.mean(pose_sims)

    # --- Component 2b: Velocity Coherence (S_velocity) ---
    # We will analyze the velocity of the primary action joint: the wrist
    wrist_exp = expert_aligned[:, 2]
    wrist_user = user_aligned[:, 2]
    
    vel_exp = np.diff(wrist_exp, axis=0, prepend=wrist_exp[0:1])
    vel_user = np.diff(wrist_user, axis=0, prepend=wrist_user[0:1])
    
    vel_sims = []
    for i in range(len(vel_exp)):
        dir_sim = cosine_similarity(vel_exp[i], vel_user[i])
        mag_diff_sq = (np.linalg.norm(vel_exp[i]) - np.linalg.norm(vel_user[i]))**2
        mag_sim = np.exp(-alpha * mag_diff_sq)
        vel_sims.append(dir_sim * mag_sim)
    s_velocity = np.mean(vel_sims)
    
    # --- Component 2c: Acceleration Profile (S_acceleration) ---
    accel_exp = np.diff(vel_exp, axis=0, prepend=vel_exp[0:1])
    accel_user = np.diff(vel_user, axis=0, prepend=vel_user[0:1])
    
    accel_sims = []
    for i in range(len(accel_exp)):
        mag_diff_sq = (np.linalg.norm(accel_exp[i]) - np.linalg.norm(accel_user[i]))**2
        accel_sims.append(np.exp(-beta * mag_diff_sq))
    s_acceleration = np.mean(accel_sims)

    # --- Final KSI Score ---
    ksi_score = (weights['pose'] * s_pose +
                 weights['velocity'] * s_velocity +
                 weights['acceleration'] * s_acceleration)

    return {
        'ksi_total': ksi_score,
        'pose_similarity': s_pose,
        'velocity_coherence': s_velocity,
        'acceleration_profile': s_acceleration
    }


# --- 3. MediaPipe Landmark Extraction ---
def extract_3d_landmarks_from_video(video_path):
    """
    Processes a video file to extract 3D pose landmarks for a single person.
    """
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=2, enable_segmentation=True, min_detection_confidence=0.5)
    
    cap = cv2.VideoCapture(video_path)
    all_landmarks = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Convert the BGR image to RGB
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        
        if results.pose_world_landmarks:
            # For simplicity, we assume one person and take their landmarks.
            # In a real scenario, you would need tracking logic to identify the same person across frames.
            landmarks = results.pose_world_landmarks.landmark
            frame_landmarks = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
            all_landmarks.append(frame_landmarks)
            
    cap.release()
    pose.close()
    return np.array(all_landmarks)


# --- 4. Example Use Case ---
if __name__ == "__main__":
    # Define paths to your videos
    EXPERT_VIDEO_PATH = "expert_smash.mp4"
    USER_VIDEO_PATH = "user_smash.mp4"
    
    print("Processing expert video...")
    expert_landmarks = extract_3d_landmarks_from_video(EXPERT_VIDEO_PATH)
    
    print("Processing user video...")
    user_landmarks = extract_3d_landmarks_from_video(USER_VIDEO_PATH)

    if expert_landmarks.shape[0] == 0 or user_landmarks.shape[0] == 0:
        print("Could not detect poses in one or both videos.")
    else:
        # Define the indices for the right arm (shoulder, elbow, wrist)
        # These are standard MediaPipe indices.
        RIGHT_SHOULDER = 12
        RIGHT_ELBOW = 14
        RIGHT_WRIST = 16
        
        # Isolate the elbow-related keypoints for analysis
        expert_elbow_seq = expert_landmarks[:, [RIGHT_SHOULDER, RIGHT_ELBOW, RIGHT_WRIST], :]
        user_elbow_seq = user_landmarks[:, [RIGHT_SHOULDER, RIGHT_ELBOW, RIGHT_WRIST], :]

        print("\nCalculating Kinetic Similarity Index (KSI) for the elbow movement...")
        ksi_results = calculate_ksi(expert_elbow_seq, user_elbow_seq)
        
        print("\n--- Analysis Results ---")
        print(f"Overall KSI Score: {ksi_results['ksi_total']:.2f}")
        print(f"  - Postural Similarity: {ksi_results['pose_similarity']:.2f}")
        print(f"  - Velocity Coherence: {ksi_results['velocity_coherence']:.2f}")
        print(f"  - Acceleration Profile: {ksi_results['acceleration_profile']:.2f}")
        
        # --- 5. Rule-Based Feedback System ("Correction Dataset" in action) ---
        print("\n--- Corrective Feedback ---")
        feedback_given = False
        
        # Rule for Pose
        if ksi_results['pose_similarity'] < 0.60:
            print("- Elbow Movement is inaccurate: Your arm's shape and angle are significantly different from the expert.")
            feedback_given = True
        elif ksi_results['pose_similarity'] < 0.80:
            print("- Elbow Movement needs improvement: Try to match the expert's arm angle more closely throughout the swing.")
            feedback_given = True
            
        # Rule for Velocity
        if ksi_results['velocity_coherence'] < 0.70:
            print("- Swing Speed: Your arm swing is either too slow or not following the correct path. Focus on a fluid motion.")
            feedback_given = True
            
        # Rule for Acceleration
        if ksi_results['acceleration_profile'] < 0.65:
            print("- Power Generation: You're losing power. Focus on accelerating the racket rapidly just before contact.")
            feedback_given = True
            
        if not feedback_given:
            print("Great form! Your elbow movement is very similar to the expert.")

In [4]:
 #--- 3. MediaPipe Landmark Extraction ---
def extract_3d_landmarks_from_video(video_path):
    """
    Processes a video file to extract 3D pose landmarks for a single person.
    """
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=2, enable_segmentation=True, min_detection_confidence=0.5)
    
    cap = cv2.VideoCapture(video_path)
    all_landmarks = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Convert the BGR image to RGB
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        
        if results.pose_world_landmarks:
            # For simplicity, we assume one person and take their landmarks.
            # In a real scenario, you would need tracking logic to identify the same person across frames.
            landmarks = results.pose_world_landmarks.landmark
            frame_landmarks = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
            all_landmarks.append(frame_landmarks)
            
    cap.release()
    pose.close()
    return np.array(all_landmarks)




In [5]:
# --- 4. Example Use Case ---
if __name__ == "__main__":
    # Define paths to your videos
    EXPERT_VIDEO_PATH = "expert_smash.mp4"
    USER_VIDEO_PATH = "user_smash.mp4"
    
    print("Processing expert video...")
    expert_landmarks = extract_3d_landmarks_from_video(EXPERT_VIDEO_PATH)
    
    print("Processing user video...")
    user_landmarks = extract_3d_landmarks_from_video(USER_VIDEO_PATH)

    if expert_landmarks.shape[0] == 0 or user_landmarks.shape[0] == 0:
        print("Could not detect poses in one or both videos.")
    else:
        # Define the indices for the right arm (shoulder, elbow, wrist)
        # These are standard MediaPipe indices.
        RIGHT_SHOULDER = 12
        RIGHT_ELBOW = 14
        RIGHT_WRIST = 16
        
        # Isolate the elbow-related keypoints for analysis
        expert_elbow_seq = expert_landmarks[:, [RIGHT_SHOULDER, RIGHT_ELBOW, RIGHT_WRIST], :]
        user_elbow_seq = user_landmarks[:, [RIGHT_SHOULDER, RIGHT_ELBOW, RIGHT_WRIST], :]

        print("\nCalculating Kinetic Similarity Index (KSI) for the elbow movement...")
        ksi_results = calculate_ksi(expert_elbow_seq, user_elbow_seq)
        
        print("\n--- Analysis Results ---")
        print(f"Overall KSI Score: {ksi_results['ksi_total']:.2f}")
        print(f"  - Postural Similarity: {ksi_results['pose_similarity']:.2f}")
        print(f"  - Velocity Coherence: {ksi_results['velocity_coherence']:.2f}")
        print(f"  - Acceleration Profile: {ksi_results['acceleration_profile']:.2f}")
        
        # --- 5. Rule-Based Feedback System ("Correction Dataset" in action) ---
        print("\n--- Corrective Feedback ---")
        feedback_given = False
        
        # Rule for Pose
        if ksi_results['pose_similarity'] < 0.60:
            print("- Elbow Movement is inaccurate: Your arm's shape and angle are significantly different from the expert.")
            feedback_given = True
        elif ksi_results['pose_similarity'] < 0.80:
            print("- Elbow Movement needs improvement: Try to match the expert's arm angle more closely throughout the swing.")
            feedback_given = True
            
        # Rule for Velocity
        if ksi_results['velocity_coherence'] < 0.70:
            print("- Swing Speed: Your arm swing is either too slow or not following the correct path. Focus on a fluid motion.")
            feedback_given = True
            
        # Rule for Acceleration
        if ksi_results['acceleration_profile'] < 0.65:
            print("- Power Generation: You're losing power. Focus on accelerating the racket rapidly just before contact.")
            feedback_given = True
            
        if not feedback_given:
            print("Great form! Your elbow movement is very similar to the expert.")

Processing expert video...


I0000 00:00:1758736257.282137  178036 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758736257.319560  180693 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1758736257.361375  180673 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1758736257.416118  180690 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1758736257.441929  178036 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758736257.479583  180722 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758736257.512839  180694 inferenc

Processing user video...
Could not detect poses in one or both videos.


W0000 00:00:1758736257.573527  180716 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
